# EV3 Notebook 2: Escalamiento y Codificacion de Variables
## Bank Marketing Dataset

Transformacion de variables numericas con StandardScaler y codificacion de variables categoricas con One-Hot Encoding.

**Anterior:** `EV3_01_Limpieza_Imputacion.ipynb`  
**Siguiente:** `EV3_03_Ingenieria_Caracteristicas.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('graficos', exist_ok=True)

# Cargar dataset
df = pd.read_csv('bank-additional-full.csv', sep=';')
df = df.drop_duplicates()

# Descartar 'duration' por ser data leaker
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])

# Renombrar columnas al espanol
rename_columns = {
    'y': 'deposito_plazo', 'age': 'edad', 'job': 'trabajo',
    'marital': 'estado_civil', 'education': 'educacion', 'default': 'mora',
    'housing': 'vivienda', 'loan': 'prestamo', 'contact': 'contacto',
    'month': 'mes', 'day_of_week': 'dia_de_la_semana', 'campaign': 'campana',
    'pdays': 'dias_previos', 'previous': 'anterior', 'poutcome': 'resultado_anterior',
    'emp.var.rate': 'var_empleo', 'cons.price.idx': 'indice_precios',
    'cons.conf.idx': 'indice_confianza', 'euribor3m': 'tasa_euribor',
    'nr.employed': 'num_empleados'
}
df = df.rename(columns=rename_columns)
df['deposito_plazo'] = df['deposito_plazo'].map({'yes': 'si', 'no': 'no'})
df['deposito_plazo_num'] = df['deposito_plazo'].map({'si': 1, 'no': 0})
df_ml = df.copy()

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")

In [ ]:
# Seccion 1 aplicada: imputacion de unknowns por moda
cols_unknown = ['trabajo', 'estado_civil', 'educacion', 'mora', 'vivienda', 'prestamo']
for col in cols_unknown:
    df_ml[col] = df_ml[col].replace('unknown', df_ml[df_ml[col] != 'unknown'][col].mode()[0])
print("Imputacion aplicada.")

## Seccion 2: Escalamiento de Variables Numericas

El escalamiento es necesario antes de aplicar algoritmos como Regresion Logistica, donde la magnitud de las variables afecta el peso de los coeficientes.

**Paso 1: Capping de outliers (IQR)**  
Se recortan valores extremos usando el rango intercuartil antes de escalar. Variables afectadas: `edad`, `campana`.

**Paso 2: StandardScaler (Z-score)**  
Transforma cada variable para tener media = 0 y desviacion estandar = 1.

Se prefiere StandardScaler sobre MinMaxScaler porque es mas robusto ante outliers residuales y evita que variables de gran escala como `num_empleados` dominen sobre las de menor rango como `anterior`.

In [ ]:
# Grafico: distribucion de variables numericas antes del escalamiento
from sklearn.preprocessing import StandardScaler

numeric_cols_ml = ['edad', 'campana', 'dias_previos', 'anterior',
                   'var_empleo', 'indice_precios', 'indice_confianza',
                   'tasa_euribor', 'num_empleados']

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols_ml):
    sns.boxplot(y=df_ml[col], ax=axes[i], color='#74b9ff', width=0.5,
                flierprops=dict(marker='o', markersize=3, alpha=0.3))
    axes[i].set_title(col)
    axes[i].set_ylabel('Valor original')
plt.suptitle('Distribucion de Variables Numericas antes del Escalamiento', fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/escalamiento_boxplots_antes.png', bbox_inches='tight')
plt.show()

In [ ]:
# Capping de outliers con IQR y aplicacion de StandardScaler
def cap_iqr(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    return s.clip(lower=Q1 - 1.5*(Q3-Q1), upper=Q3 + 1.5*(Q3-Q1))

for col in ['edad', 'campana']:
    antes = df_ml[col].max()
    df_ml[col] = cap_iqr(df_ml[col])
    print(f"{col}: max antes = {antes:.1f}, max despues = {df_ml[col].max():.1f}")

scaler = StandardScaler()
df_ml[numeric_cols_ml] = scaler.fit_transform(df_ml[numeric_cols_ml])
print()
print("Media y desviacion estandar post-escalamiento:")
print(df_ml[numeric_cols_ml].describe().loc[['mean', 'std']].round(4).to_string())

In [ ]:
# Grafico: distribucion de variables numericas despues del escalamiento
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols_ml):
    sns.boxplot(y=df_ml[col], ax=axes[i], color='#55efc4', width=0.5,
                flierprops=dict(marker='o', markersize=3, alpha=0.3))
    axes[i].set_title(col)
    axes[i].set_ylabel('Valor escalado (Z-score)')
plt.suptitle('Distribucion de Variables Numericas despues del Escalamiento (StandardScaler)', fontweight='bold')
plt.tight_layout()
plt.savefig('graficos/escalamiento_boxplots_despues.png', bbox_inches='tight')
plt.show()

## Seccion 3: Codificacion de Variables Categoricas

Las variables categoricas no pueden ser procesadas directamente por los algoritmos de ML. Se aplica One-Hot Encoding.

**Tecnica: One-Hot Encoding con `drop_first=True`**

| Variable | Tipo | Razon para OHE |
|---|---|---|
| trabajo | Nominal (11 categorias) | Sin orden entre profesiones |
| estado_civil | Nominal (3 categorias) | Sin jerarquia |
| educacion | Ordinal* | OHE evita asumir distancias iguales entre niveles |
| mora, vivienda, prestamo | Binaria (2 categorias) | Con drop_first equivale a 0/1 |
| contacto | Nominal (2 categorias) | Celular vs telefono fijo |
| mes, dia_de_la_semana | Ciclico nominal | Sin orden lineal entre meses o dias |
| resultado_anterior | Nominal (3 categorias) | Sin jerarquia |

`drop_first=True` elimina la primera categoria de cada variable para evitar multicolinealidad perfecta en la Regresion Logistica.

In [ ]:
# One-Hot Encoding de variables categoricas
cat_cols_ml = df_ml.select_dtypes(include=['object']).columns.drop('deposito_plazo').tolist()

print("Variables a codificar:", cat_cols_ml)
print(f"Shape antes OHE: {df_ml.shape}")

df_ml = pd.get_dummies(df_ml, columns=cat_cols_ml, drop_first=True)
bool_cols = df_ml.select_dtypes(include=['bool']).columns
df_ml[bool_cols] = df_ml[bool_cols].astype(int)

print(f"Shape despues OHE: {df_ml.shape}")
print(f"{df_ml.shape[1] - 1} predictores + 1 target (deposito_plazo_num)")